# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {'A': 2, 'B': 1}, код поочередно добавляет каждый элемент списка в словарь, пока не закончатся элементы списка через присвоение, то есть метод get безопасно достает первый элемент списка "А", но в словаре такого ключа пока нет по этому он возвращает ноль и далее +1 получается  пока r = {"A": 1} далее таким же методом добавляется второй элемент затем попадается повторяющиеся ключ который уже есть в списке и вместо нуля он уже к существующему элементу А добавляет +1 и получается {'A': 2, 'B': 1}
2. [0, 1 ,1], создается функция которая в конец списка добавляет элемент (1) через .append, в списке имеется только один элемент a = [0], далее функция вызывается два раз и в конец списка добавлется еще две однерки
3. код выведет значение: 2, фигурными скобками был задан set comprehension который переводит все элементы списка в нижний  регистр чтобы сделать их одинаковыми, далее вызывается функция len которая считает количество элементов так как set сохраняет только уникальные значение в множество выйдет 2
4. {'ab': 2, 'abc': 3}, задается dict comprehension в котором слева ключ k(элемент списка), а справа количество символов в элементе списка через len(), так получается, что ab => 2 abc => 3, послдений элемент перезаписывает значение уже имеющегося ключа а не добавляет новый, так как ключи в словаре всегда уникальны 
5. 5
   None, задается функция которая сравнивает больше ли значение 0, если да то оно выводится на экран через print, выводится на экран значение 5, но ни куда не возвращается так как нет return, следующий прринт ниже выведет None так как функция ничего не возвращает 

In [4]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [7]:
# База
#1
def to_num(s):
    if not s or not s.strip():
        return 0.0
    return float(s.replace(" ", ""))

def clean_city(s):
    mapping = {"almaty": "Алматы","astana": "Астана","shymkent": "Шымкент","atyrau": "Атырау","karaganda": "Караганда","aktobe": "Актобе"}
    city_clean = s.strip().lower()
    return mapping.get(city_clean, s)

clean_city(rows[0]["city"])
to_num(rows[0]["amount"])

8070.0

In [10]:
#2
r ={}
for row in rows:
    categ = row["category"]
    r[categ] = r.get(categ,0) + 1

sortirovka = dict(sorted(r.items(), key = lambda x: x[1], reverse = True))

print(sortirovka) 

{'Одежда': 1529, 'Электроника': 1185, 'Дом': 1105, 'Красота': 991, 'Книги': 653}


In [32]:
#3
category_revenue = {}
category_count = {}

for row in rows:
    if row["status"] == "оплачен":
        c = row["category"]
        r = to_num(row["amount"])
        
        category_revenue[c] = category_revenue.get(c, 0.0) + r
        category_count[c] = category_count.get(c, 0) + 1
for c in category_revenue:
    rev = category_revenue[c]
    coun = category_count[c]
    avg = rev/coun
    print(f"Выручка: | {c} | {rev} | Средний чек: {avg}")

    

Выручка: | Одежда | 18321080.0 | Средний чек: 13285.772298767222
Выручка: | Красота | 5507620.0 | Средний чек: 6188.337078651685
Выручка: | Дом | 10516320.0 | Средний чек: 10654.83282674772
Выручка: | Книги | 2294100.0 | Средний чек: 3934.991423670669
Выручка: | Электроника | 65270240.0 | Средний чек: 61286.6103286385


In [30]:
#4
months_count = {}

for row in rows:
    month = row["order_date"][:7] 
    months_count[month] = months_count.get(month, 0) + 1

print(months_count)

{'2026-09': 2782, '2026-10': 2681}


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж